# Sabahattin Ali Metinleri ile Char-Level GPT Modeli

Bu projede, Sabahattin Ali'nin eserlerinden oluşturulan bir veri seti kullanılarak karakter seviyesinde çalışan bir üretken yapay zeka modeli geliştirilmiştir. Modelin amacı, verilen bir başlangıç cümlesinden sonra benzer üslupta yeni metinler üretebilmektir.

## 1. Projenin Amacı

Bu projede amaç, karakter seviyesinde çalışan bir dil modeli oluşturarak Türkçe edebi metin üretimi yapabilmektir. Model, kelime veya cümle bazında değil, doğrudan karakter dizileri üzerinden öğrenme yapmaktadır.

Bu yaklaşım sayesinde model, dilin yapısını (harfler, boşluklar, noktalama işaretleri) öğrenebilir. Ancak anlam düzeyinde bazı sınırlamalar oluşması beklenmektedir.

Veri seti olarak Sabahattin Ali'nin eserleri kullanılmıştır. Eğitim sonunda modelden, verilen bir başlangıç ifadesine benzer şekilde yeni metinler üretmesi beklenmiştir.

## 2. Proje Yapısı

Projede kullanılan temel dosyalar aşağıdaki gibidir:

- `clean_dataset.py` → Ham metinlerin temizlenmesi
- `prepare_corpus.py` → Veri setinin hazırlanması ve train/val ayrımı
- `train_char_gpt.py` → Modelin eğitilmesi
- `generate.py` → Eğitilen model ile metin üretimi

Ek olarak:

- `train.txt` → Eğitim verisi
- `val.txt` → Doğrulama verisi
- `char_gpt_best.pt` → En iyi model (validation loss’a göre seçilmiş)

In [ ]:
from pathlib import Path

base = Path(".")
print("Proje klasörü:", base.resolve())

for folder in ["code", "data", "model"]:
    print(folder, "var mı?", (base / folder).exists())

## 3. Veri Hazırlama

Projede kullanılan metinler öncelikle temizlenmiştir. Bu aşamada gereksiz boşluklar, bozuk karakterler ve metni olumsuz etkileyebilecek bazı yapılar düzeltilmiştir.

Daha sonra tüm metinler tek bir corpus haline getirilmiş ve kitap sınırlarını belirlemek için özel etiketler eklenmiştir.

Hazırlanan veri seti yaklaşık olarak %90 eğitim ve %10 doğrulama olacak şekilde ikiye ayrılmıştır.

In [ ]:
from pathlib import Path

train_text = Path("data/train.txt").read_text(encoding="utf-8", errors="replace")
val_text = Path("data/val.txt").read_text(encoding="utf-8", errors="replace")

print("Train karakter sayısı:", len(train_text))
print("Validation karakter sayısı:", len(val_text))
print("Toplam karakter sayısı:", len(train_text) + len(val_text))
print("Vocab karakter sayısı:", len(set(train_text + val_text)))

## 4. Model Mimarisi

Projede GPT benzeri, decoder-only bir Transformer modeli kullanılmıştır. Model tamamen karakter seviyesinde çalışmaktadır.

Bu model:

- Karakterleri token olarak kullanır
- Önceki karakterlere bakarak bir sonraki karakteri tahmin eder
- Self-attention mekanizması ile bağlam öğrenir

Ancak model kelime anlamlarını doğrudan öğrenmediği için, uzun metinlerde anlam kopmaları görülebilir.

## 5. Eğitim Parametreleri

Model aşağıdaki temel parametrelerle eğitilmiştir:

- Block size: 256
- Dropout: 0.10
- Learning rate: 2e-4
- Iterasyon sayısı: 15000
- Temperature: 0.7
- Top-k: 40

İlk denemelerde modelin çıktılarında fazla karakter hatası bulunuyordu. Bu nedenle bazı parametreler değiştirilerek modelin daha stabil öğrenmesi sağlandı.

## 6. Eğitim Süreci

Model 15000 iterasyon boyunca eğitilmiştir.

Eğitim sırasında validation loss değeri takip edilerek en iyi model ayrı olarak kaydedilmiştir (`char_gpt_best.pt`).

Final model yerine bu checkpoint tercih edilmiştir. Çünkü eğitim ilerledikçe modelin overfit olma ihtimali artmaktadır.

## 7. Eğitim Sonuçları

Eğitim sonunda elde edilen değerler:

- Train loss: 0.6973
- Validation loss: 1.6774

Train loss’un düşük olması modelin veriyi iyi öğrendiğini gösterirken, validation loss’un daha yüksek olması modelin sınırlı veri nedeniyle tam genelleme yapamadığını göstermektedir.

Buna rağmen, üretilen metinlerin kalitesi önceki denemelere göre belirgin şekilde iyileşmiştir.

## 8. Metin Üretimi

Model ile metin üretimi `generate.py` dosyası kullanılarak yapılmıştır.

Kullanılan ayarlar:

- Prompt: "Macide o akşam"
- Temperature: 0.7
- Top-k: 40
- Yeni karakter sayısı: 500

## 9. Üretilen Metin

Macide o akşam daha lüzumlu ve siyah, muhakkak oluyordu. Bu sırada annesine, onun sebebini biraz tereddüt ederek baktı.

“Yani... bilmem,” dedi. “Hattâ benim daha olmadım... Biraz bu düşündürdüğünü, nereye olursa olsun, bunların ne olduğunu bile bilmiyorum. Senin neden ayrılmaya başladığımı, manasız bir takdir içinde olduğumu, akşam evlerini düşünerek bu kadar sabit bir huzursuzluk olduğunu fark ediyorum. Sonra korkunç bir şeyin içinde, adamakıllı görünmekten korkuyordum.”

Bir an durdu. İlk defa böyle konuşmak ister gibi oldu, fakat sözleri yarım kaldı.

(**küçük düzenlemeler yapılmıştır**)


## 10. Değerlendirme

Model, Türkçe karakter yapısını ve genel cümle akışını başarılı bir şekilde öğrenmiştir. Özellikle kelime oluşumları ve cümle yapısı önceki denemelere göre daha tutarlıdır.

Ancak model karakter seviyesinde çalıştığı için anlam bütünlüğü her zaman korunamamaktadır. Uzun metinlerde tekrarlar ve anlamsal kopmalar görülebilir.

Ayrıca bu projede önemli bir gözlem yapılmıştır: Validation loss daha düşük olan model her zaman daha iyi metin üretmemektedir. Bu nedenle model seçimi sadece loss değerine göre değil, üretilen çıktının kalitesine göre yapılmıştır.

## 11. Sonuç

Bu projede Sabahattin Ali eserleri kullanılarak char-level bir dil modeli geliştirilmiştir.

Model, Türkçe metin üretiminde belirli bir başarı göstermiştir. Yapılan iyileştirmeler sayesinde karakter hataları azaltılmış ve daha okunabilir metinler elde edilmiştir.

Ancak veri setinin sınırlı olması ve modelin karakter seviyesinde çalışması nedeniyle anlam bütünlüğü tamamen sağlanamamıştır.

Genel olarak proje, üretken yapay zeka ve dil modeli mantığını anlamak açısından başarılı bir çalışma olmuştur.

## 12. Kaynaklar
1.   Vaswani, A., et al. (2017). Attention Is All You Need.

2.   Radford, A., et al. (2019). Language Models are Unsupervised Multitask Learners (GPT-2 Paper).

3.   Karpathy, A. (2022). Let's build GPT: from scratch, in code.